In [0]:
# Importing libraries
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("file_name", "")

In [0]:
data_file_name = dbutils.widgets.get("file_name")

In [0]:
data_file_name

## **Reading data using autoloader**

In [0]:
df_orders = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "csv")\
                .option("header", "true")\
                .option("cloudFiles.schemaLocation", f"/Volumes/ecommerce_project/bronze_ecom/checkpoint_{data_file_name}")\
                .load(f"/Volumes/ecommerce_project/default/data_landing/{data_file_name}/")\
                .withColumn("_ingest_time", current_timestamp())\
                .withColumn("_file_path", col("_metadata.file_path"))

## **Writing data**

In [0]:
df_orders.writeStream.format("delta")\
    .option("checkpointLocation", f"/Volumes/ecommerce_project/bronze_ecom/checkpoint_{data_file_name}")\
    .option("path", f"/Volumes/ecommerce_project/bronze_ecom/{data_file_name}")\
    .outputMode("append")\
    .trigger(once=True)\
    .start()

In [0]:
spark.read.format("delta") \
    .load(f"/Volumes/ecommerce_project/bronze_ecom/{data_file_name}") \
    .count()
